# React 框架 · 第 03 天

## Redux、Redux Toolkit 与 React-Redux

本篇依据已下载的 React 基础 day03 MD 笔记、PPT 和随堂代码整理。内容从 Redux 的基本数据流开始，学习 Redux Toolkit、React-Redux、同步与异步 action，并通过美团外卖案例理解商品分类、购物车和统计区域的集中状态管理。

## 一、学习目标

学完本篇后，应能：

- 说明为什么需要集中式状态管理，以及 Redux 解决什么问题。
- 理解 `state`、`action`、`reducer` 和 `store` 的关系。
- 用纯 Redux 完成一个计数器，理解 dispatch 到视图更新的完整链路。
- 使用 Redux Toolkit 的 `createSlice` 和 `configureStore` 编写模块化状态。
- 使用 `Provider`、`useSelector`、`useDispatch` 把 Redux 接入 React。
- 给 action creator 传递 payload，并处理异步请求。
- 使用 Redux DevTools 调试 state 和 action。
- 用 Redux 管理美团案例中的商品列表、激活分类和购物车。

## 二、为什么使用 Redux

Redux 是常用的集中状态管理工具，可以独立于 React 运行，也可以通过 `react-redux` 与 React 组件连接。它把跨组件、跨页面共享的数据放在统一的 store 中管理。

### 适合使用 Redux 的场景

- 多个组件共享同一份数据。
- 组件层级很深，逐层传 props 造成通信负担。
- 数据修改入口较多，需要统一规则。
- 需要记录 action、查看状态变化或回放调试。

### 主要优势

1. **和组件层级解耦**：共享状态不依赖组件之间的直接嵌套关系。
2. **单向数据流清晰**：组件 dispatch action，reducer 产生新 state，视图重新渲染。
3. **调试工具完善**：可以查看每次 action 和前后 state。
4. **生态成熟**：Redux Toolkit、React-Redux 和各种中间件提供了完整方案。

局部、短生命周期的状态仍然优先放在组件自身；Redux 适合跨组件共享的业务状态。

## 三、Redux 的核心概念与数据流

Redux 的核心可以归纳为 4 个概念：

- **state**：一个对象，保存应用要集中管理的数据。
- **action**：一个描述“想做什么”的普通对象，必须包含 `type`。
- **reducer**：根据当前 state 和 action，计算并返回下一个 state。
- **store**：保存 state，并提供 `dispatch`、`getState`、`subscribe` 等方法。

完整数据流：

```text
用户操作
  ↓
组件 dispatch(action)
  ↓
reducer(state, action)
  ↓
得到新的 state
  ↓
订阅者收到通知
  ↓
组件重新渲染
```

组件不能直接修改 store，必须通过 dispatch 提交 action；reducer 负责根据 action 描述完成状态变化。

## 四、纯 Redux 计数器

在不依赖 React 和构建工具的情况下，也可以直接使用 Redux：

```html
<button id="decrement">-</button>
<span id="count">0</span>
<button id="increment">+</button>
<script src="https://unpkg.com/redux@latest/dist/redux.min.js"></script>
<script>
function counterReducer(state = { count: 0 }, action) {
  switch (action.type) {
    case 'INCREMENT':
      return { count: state.count + 1 }
    case 'DECREMENT':
      return { count: state.count - 1 }
    default:
      return state
  }
}

const store = Redux.createStore(counterReducer)
const render = () => {
  document.querySelector('#count').textContent = store.getState().count
}

store.subscribe(render)
render()
document.querySelector('#increment').addEventListener('click', () => {
  store.dispatch({ type: 'INCREMENT' })
})
document.querySelector('#decrement').addEventListener('click', () => {
  store.dispatch({ type: 'DECREMENT' })
})
</script>
```

实现步骤：

1. 定义 reducer，描述每种 action 对 state 的修改。
2. 使用 `createStore(reducer)` 创建 store。
3. 使用 `subscribe` 订阅数据变化。
4. 使用 `dispatch(action)` 触发修改。
5. 使用 `getState()` 读取最新数据并渲染视图。

Redux 的难点不是 API 数量，而是理解“组件不能越过 dispatch 直接改状态”的数据流约束。

## 五、Redux Toolkit 与 React-Redux

在 React 项目中，官方推荐使用两套配套工具：

- `@reduxjs/toolkit`（RTK）：简化 reducer、action 和 store 的编写。
- `react-redux`：通过 Provider 和 Hooks 把 Redux 与 React 组件连接起来。

安装：

```bash
npx create-react-app react-redux
cd react-redux
npm i @reduxjs/toolkit react-redux
npm run start
```

建议集中创建 `src/store` 目录：

```text
src/
├─ store/
│  ├─ index.js          # 组合所有子模块并导出 store
│  └─ modules/
│     ├─ counterStore.js
│     └─ channelStore.js
├─ App.js
└─ index.js
```

按业务划分子模块，可以避免所有状态写在一个巨大文件中。

## 六、用 `createSlice` 创建 counter 模块

`createSlice` 会根据 `reducers` 自动生成 reducer 和 action creator：

```js
// store/modules/counterStore.js
import { createSlice } from '@reduxjs/toolkit'

const counterStore = createSlice({
  name: 'counter',
  initialState: { count: 1 },
  reducers: {
    increment(state) {
      state.count++
    },
    decrement(state) {
      state.count--
    },
  },
})

export const { increment, decrement } = counterStore.actions
export default counterStore.reducer
```

虽然代码看起来直接修改了 `state.count`，但 Redux Toolkit 内部使用 Immer，会把这段“可变写法”转换为安全的不可变更新。模块的 `name` 应保持唯一，`initialState` 应提供完整且稳定的数据结构。

## 七、组合 store 并注入 React

使用 `configureStore` 注册各个子 reducer：

```js
// store/index.js
import { configureStore } from '@reduxjs/toolkit'
import counterReducer from './modules/counterStore'

const store = configureStore({
  reducer: {
    counter: counterReducer,
  },
})

export default store
```

在应用入口通过 `Provider` 把 store 注入组件树：

```jsx
import ReactDOM from 'react-dom/client'
import { Provider } from 'react-redux'
import App from './App'
import store from './store'

ReactDOM.createRoot(document.getElementById('root')).render(
  <Provider store={store}>
    <App />
  </Provider>
)
```

Provider 负责让后代组件能够访问同一个 store。没有 Provider 时，在组件中调用 `useSelector` 或 `useDispatch` 会报错。

## 八、组件读取和修改 store

### `useSelector` 读取状态

```jsx
import { useSelector } from 'react-redux'

function Counter() {
  const count = useSelector(state => state.counter.count)
  return <p>当前值：{count}</p>
}
```

### `useDispatch` 提交 action

```jsx
import { useDispatch } from 'react-redux'
import { increment, decrement } from './store/modules/counterStore'

function Counter() {
  const dispatch = useDispatch()
  return (
    <>
      <button onClick={() => dispatch(decrement())}>-</button>
      <button onClick={() => dispatch(increment())}>+</button>
    </>
  )
}
```

`useSelector` 建立组件与指定状态的订阅关系，相关状态变化后组件会重新渲染；`useDispatch` 返回提交 action 的函数。选择器尽量只读取组件真正需要的字段。

## 九、action 传参：`payload`

需求：点击不同按钮，让 count 直接增加到指定数字。可以在 action creator 调用时传参，参数会进入 action 的 `payload`：

```js
const counterStore = createSlice({
  name: 'counter',
  initialState: { count: 1 },
  reducers: {
    setCount(state, action) {
      state.count = action.payload
    },
  },
})

export const { setCount } = counterStore.actions
```

```jsx
const dispatch = useDispatch()

<button onClick={() => dispatch(setCount(10))}>加到 10</button>
<button onClick={() => dispatch(setCount(20))}>加到 20</button>
```

传递多个值时可以使用对象：

```jsx
dispatch(updateUser({ id: 1, name: 'rose' }))
```

```js
updateUser(state, action) {
  state.user = action.payload
}
```

## 十、异步 action：请求数据后再 dispatch

Redux Toolkit 的同步 reducer 只描述状态如何改变，异步请求可以封装成一个返回函数的 action：

```js
import { createSlice } from '@reduxjs/toolkit'
import axios from 'axios'

const channelStore = createSlice({
  name: 'channel',
  initialState: { channelList: [] },
  reducers: {
    setChannelList(state, action) {
      state.channelList = action.payload
    },
  },
})

export const { setChannelList } = channelStore.actions

export const fetchChannelList = () => {
  return async dispatch => {
    const response = await axios.get(
      'http://geek.itheima.net/v1_0/channels'
    )
    dispatch(setChannelList(response.data.data.channels))
  }
}

export default channelStore.reducer
```

组件中的调用方式和同步 action 一样：

```jsx
const dispatch = useDispatch()
const { channelList } = useSelector(state => state.channel)

useEffect(() => {
  dispatch(fetchChannelList())
}, [dispatch])
```

异步逻辑负责请求和整理数据，成功后 dispatch 同步 action；组件只关心触发动作和展示状态。实际项目还应补充 loading、error 和请求取消处理。

## 十一、Redux DevTools

Redux DevTools 可以查看：

- 当前完整 state。
- 每次 dispatch 的 action 类型和 payload。
- action 前后的状态差异。
- 状态变化时间线，必要时进行回放。

排查 Redux 问题时，建议按以下顺序：

1. 事件是否触发。
2. dispatch 的 action 类型和 payload 是否正确。
3. reducer 是否被正确注册到 store。
4. reducer 是否把数据写入预期路径。
5. `useSelector` 的选择器路径是否正确。
6. 组件是否因为读取了新状态而重新渲染。

DevTools 负责观察数据流，不能替代清晰的模块边界和正确的状态设计。

## 十二、美团案例：总体拆分

案例使用 Redux Toolkit 管理外卖页面状态，组件负责渲染和 dispatch action。建议先准备：

```bash
git clone http://git.itcast.cn/heimaqianduan/redux-meituan.git
cd redux-meituan
npm i
npm run serve
npm run start
```

主要状态可以按业务拆分：

- `foodsList`：分类和商品数据。
- `activeIndex`：当前激活的商品分类。
- `cartList`：购物车商品和数量。
- 购物车显示状态：可以保留在 `Cart` 组件内部，因为它是局部 UI 状态。

把业务数据放入 store，把纯界面显示状态留在组件中，是集中状态和局部状态的常见边界。

## 十三、美团案例：分类与商品列表

### store 中请求商品分类

```js
const foodsStore = createSlice({
  name: 'foods',
  initialState: { foodsList: [], activeIndex: 0 },
  reducers: {
    setFoodsList(state, action) {
      state.foodsList = action.payload
    },
    changeActiveIndex(state, action) {
      state.activeIndex = action.payload
    },
  },
})

export const { setFoodsList, changeActiveIndex } = foodsStore.actions

export const fetchFoodsList = () => async dispatch => {
  const response = await axios.get('http://localhost:3004/takeaway')
  dispatch(setFoodsList(response.data))
}
```

### 组件渲染菜单

```jsx
const { foodsList, activeIndex } = useSelector(state => state.foods)
const dispatch = useDispatch()

useEffect(() => {
  dispatch(fetchFoodsList())
}, [dispatch])

const menus = foodsList.map(item => ({ tag: item.tag, name: item.name }))

return menus.map((item, index) => (
  <div
    key={item.tag}
    className={activeIndex === index ? 'list-menu-item active' : 'list-menu-item'}
    onClick={() => dispatch(changeActiveIndex(index))}
  >
    {item.name}
  </div>
))
```

商品内容可以根据 `activeIndex` 只渲染当前分类：

```jsx
{foodsList.map((item, index) => (
  activeIndex === index && (
    <FoodsCategory key={item.tag} name={item.name} foods={item.foods} />
  )
))}
```

## 十四、美团案例：添加商品到购物车

购物车状态放到 foods 模块或单独的 cart 模块中均可，关键是数据结构稳定：

```js
const foodsStore = createSlice({
  name: 'foods',
  initialState: { cartList: [] },
  reducers: {
    addCart(state, action) {
      const item = state.cartList.find(
        cartItem => cartItem.id === action.payload.id
      )

      if (item) {
        item.count++
      } else {
        state.cartList.push({ ...action.payload, count: 1 })
      }
    },
    increaseCount(state, action) {
      const item = state.cartList.find(
        cartItem => cartItem.id === action.payload.id
      )
      if (item) item.count++
    },
    decreaseCount(state, action) {
      const item = state.cartList.find(
        cartItem => cartItem.id === action.payload.id
      )
      if (item && item.count > 0) item.count--
    },
    clearCart(state) {
      state.cartList = []
    },
  },
})
```

商品卡片点击加号时提交商品信息：

```jsx
<span
  className="plus"
  onClick={() => dispatch(addCart({ id, name, price, picture }))}
/>
```

RTK 的 Immer 允许 reducer 里写 `item.count++`，但组件仍然只能通过 dispatch action 修改 store。

## 十五、美团案例：购物车统计和列表

总数量、总价和高亮状态都可以从 `cartList` 计算得到，不必额外保存重复状态：

```jsx
const { cartList } = useSelector(state => state.foods)

const totalCount = cartList.reduce((sum, item) => sum + item.count, 0)
const totalPrice = cartList.reduce(
  (sum, item) => sum + item.price * item.count,
  0
)
```

```jsx
<div className={cartList.length ? 'icon fill' : 'icon'}>
  {totalCount > 0 && <span>{totalCount}</span>}
</div>
<div>合计：¥{totalPrice}</div>
```

购物车列表用 `map` 渲染，数量组件通过回调 dispatch 增减：

```jsx
{cartList.map(item => (
  <CartItem
    key={item.id}
    item={item}
    onPlus={() => dispatch(increaseCount({ id: item.id }))}
    onMinus={() => dispatch(decreaseCount({ id: item.id }))}
  />
))}
```

购物车的展开/收起是局部显示状态，可以在 `Cart` 组件中使用 `useState`；清空购物车则属于业务数据修改，应 dispatch `clearCart()`。

## 十六、集中状态与局部状态的取舍

不是所有状态都应该放进 Redux：

### 适合放 Redux

- 多个页面或多个组件共享的用户信息、商品、购物车、权限。
- 需要跨层访问或由多个模块修改的数据。
- 希望统一记录和调试的数据。

### 适合放组件

- 弹窗是否打开、输入框临时值、鼠标悬停、当前动画阶段。
- 只在一个组件内部使用、不会被其他模块读取的数据。

把状态放在“离所有使用它的组件最近、但足够共享”的位置，可以减少不必要的全局依赖和渲染。

## 十七、常见错误与排查顺序

1. **`useSelector` 取不到值**：确认 Provider 已包裹组件，且选择器路径与 `configureStore` 的 reducer key 一致。
2. **action 没有生效**：确认导入的是 slice 导出的 action creator，并且调用了 `dispatch(action())`。
3. **payload 是 undefined**：检查 action creator 调用时是否传参，reducer 是否从 `action.payload` 读取。
4. **异步请求没有执行**：检查 thunk 是否返回函数、是否使用 `dispatch(fetchData())`，并确认网络请求地址。
5. **reducer 注册错误**：在 `configureStore({ reducer: { ... } })` 中注册的是 reducer 函数，不是整个 slice 对象。
6. **直接改 store**：组件不能写 `store.state.xxx = ...`，所有修改都走 action。
7. **总价和总数不同步**：从 `cartList` 派生计算，不要同时维护容易不一致的冗余状态。
8. **商品 key 不稳定**：使用商品 id 或分类 tag，不要使用数组下标作为动态列表 key。

排查 Redux 问题时先看 DevTools 的 action，再看 reducer 状态路径，最后检查组件选择器和渲染条件。

## 十八、本日总结

- Redux 用集中 store 管理跨组件共享状态，数据流是 action → reducer → state → UI。
- `createSlice` 自动生成 reducer 和 action creator，是编写 Redux 逻辑的推荐方式。
- `configureStore` 组合子模块，`Provider` 把 store 注入 React，`useSelector` 读取，`useDispatch` 修改。
- action 参数通过 `payload` 传递，异步请求完成后再 dispatch 同步 action。
- Redux DevTools 可以追踪状态变化和 action 时间线。
- 美团案例把商品请求、分类切换、购物车增减、清空和统计串联起来。
- 全局共享数据放 Redux，局部界面状态留在组件中，能让代码边界更清晰。

下一步可以继续学习 Redux 的更复杂异步流程、持久化、路由和完整项目状态设计。